Find the files

In [0]:
from pathlib import Path

DATA_DIR_PARQ = Path(r"C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7")
DATA_DIR_CSV = Path(r"C:\Users\adiz\Downloads\archive(1)")

parquet_files = sorted(DATA_DIR_PARQ.glob("*.parquet"))
csv_files = sorted(DATA_DIR_CSV.glob("*.csv"))

print("Parquet files:")
for f in parquet_files:
    print(f)

print("\nCSV files:")
for f in csv_files:
    print(f)


Inspect the Parquet files without loading everything

In [0]:
import pandas as pd

for file in parquet_files:
    df = pd.read_parquet(file)

    print(
        f"{file.name:30s} "
        f"rows={len(df):,} "
        f"columns={len(df.columns)}"
    )


Check that all 7 files have the same columns

In [0]:
schemas = {}

for file in parquet_files:
    df = pd.read_parquet(file)

    schemas[file.name] = {
        column: str(dtype)
        for column, dtype in df.dtypes.items()
    }

for file, schema in schemas.items():
    print(f"\n{file}")
    print("-" * 60)

    for column, dtype in schema.items():
        print(f"{column:30s} {dtype}")




In [0]:
reference_file = parquet_files[0]
reference_df = pd.read_parquet(reference_file)

reference_columns = set(reference_df.columns)

for file in parquet_files[1:]:
    df = pd.read_parquet(file)

    columns = set(df.columns)

    missing_columns = reference_columns - columns
    extra_columns = columns - reference_columns

    if missing_columns or extra_columns:
        print(f"\n{file.name}")

        if missing_columns:
            print("Missing columns:", missing_columns)

        if extra_columns:
            print("Extra columns:", extra_columns)


Missing-value analysis across all 7 files

In [0]:
'''
Emplainable missing values only in two situations: the purchase is made A) online or B) outside the U.S:

In case A)
Both Zip + Merchant State are empty

In case B)
"Merchant State" has a country name + Zip is empty (as zip is only supported for U.S.)
'''

for file in parquet_files:
    df = pd.read_parquet(file)

    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100,
    })

    missing = missing[missing["missing_count"] > 0]
    missing = missing.sort_values("missing_count", ascending=False)

    print(f"\n{'=' * 80}")
    print(file.name)
    print(f"{'=' * 80}")

    display(missing)


Check the actual transaction columns

In [0]:
transactions_sample = pd.read_parquet(parquet_files[0])

print(transactions_sample.columns.tolist())


In [0]:
display(
    pd.DataFrame({
        "column": transactions_sample.columns,
        "dtype": transactions_sample.dtypes.astype(str),
        "unique": transactions_sample.nunique().values,
        "missing": transactions_sample.isna().sum().values,
        "missing_pct": (
            transactions_sample.isna().mean().values * 100
        )
    })
)


Read users and cards separately

In [0]:
users = pd.read_csv(DATA_DIR_CSV / "sd254_users.csv")
cards = pd.read_csv(DATA_DIR_CSV / "sd254_cards.csv")

print("Users:", users.shape)
print("Cards:", cards.shape)


In [0]:
print("Users columns:")
print(users.columns.tolist())

print("\nCards columns:")
print(cards.columns.tolist())


In [0]:
display(users.head())
display(cards.head())


In [0]:
print("USERS")
display(
    users.isna()
    .sum()
    .to_frame("missing")
    .assign(
        missing_pct=lambda x: x["missing"] / len(users) * 100
    )
    .sort_values("missing", ascending=False)
)

print("CARDS")
display(
    cards.isna()
    .sum()
    .to_frame("missing")
    .assign(
        missing_pct=lambda x: x["missing"] / len(cards) * 100
    )
    .sort_values("missing", ascending=False)
)


Referential-integrity check to determine whether every transaction's card/user identifier actually exists in the corresponding CSVs

In [0]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")

# Load the relatively small dimension tables
users = pd.read_csv(DATA_DIR_CSV / "sd254_users.csv")
cards = pd.read_csv(DATA_DIR_CSV / "sd254_cards.csv")

print("Users:", users.shape)
print("Cards:", cards.shape)

print("\nUsers columns:")
print(users.columns.tolist())

print("\nCards columns:")
print(cards.columns.tolist())

#Check facts table
print("\nTransactions columns:")
print(transactions_sample.columns.tolist())


In [0]:
# Check referential integrity:
# cards:        User + CARD INDEX
# transactions: User + Card

# Create the set of valid (User, CARD INDEX) combinations from cards.csv
valid_card_keys = set(
    zip(
        cards["User"],
        cards["CARD INDEX"]
    )
)

print(f"Unique valid User + CARD INDEX combinations: {len(valid_card_keys):,}")


# Check every transaction parquet file
integrity_results = []

for file in parquet_files:

    transactions = pd.read_parquet(file)

    # Create (User, Card) pairs for transactions
    transaction_keys = list(
        zip(
            transactions["User"],
            transactions["Card"]
        )
    )

    # Check whether each transaction's User + Card exists in cards
    exists_in_cards = [
        key in valid_card_keys
        for key in transaction_keys
    ]

    exists_in_cards = pd.Series(
        exists_in_cards,
        index=transactions.index
    )

    invalid_count = (~exists_in_cards).sum()

    integrity_results.append({
        "file": file.name,
        "transactions": len(transactions),
        "valid": exists_in_cards.sum(),
        "invalid": invalid_count,
        "invalid_pct": invalid_count / len(transactions) * 100
    })


# Display results
integrity_report = pd.DataFrame(integrity_results)

display(integrity_report)


This relationship can be established:

transactions.User
        +
transactions.Card
        │
        ▼
cards.User
        +
cards.CARD INDEX


check that the User and Card combination is unique in cards.csv

In [0]:
# Check whether (User, CARD INDEX) uniquely identifies a card

duplicate_card_keys = (
    cards
    .groupby(["User", "CARD INDEX"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)

print(f"Duplicate (User, CARD INDEX) combinations: {len(duplicate_card_keys):,}")

display(duplicate_card_keys.head(20))


Then we can say that (User, CARD INDEX) is a proper composite key for the cards table.

Vice versa: Card values are unique within each user?

In [0]:
print(
    cards.duplicated(
        subset=["User", "CARD INDEX"]
    ).sum()
)


users
  │
  │ User
  ▼
cards
  │
  │ User + CARD INDEX
  ▼
transactions
